# Assignment 1: Image Classification — ANN vs CNN

**Objective:** Compare the architectural and computational differences between a
traditional Artificial Neural Network (ANN / MLP) and a Convolutional Neural
Network (CNN) on a real-world computer vision task, using the
[Intel Image Classification dataset](https://www.kaggle.com/datasets/puneet6060/intel-image-classification)
(~25,000 images across 6 classes: `buildings`, `forest`, `glacier`, `mountain`, `sea`, `street`).

**Contents**
1. Setup
2. Dataset acquisition
3. Data preprocessing
4. Model architectures (ANN & CNN)
5. Training utilities
6. Train the ANN baseline
7. Train the CNN
8. Comparison analysis (accuracy, training time, parameter count)
9. Discussion — why CNNs win on spatial data
10. Bonus — inference speed & CUDA/TensorRT optimization


## 1. Setup

In [ ]:
import os
import time
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## 2. Dataset Acquisition

The Intel Image Classification dataset is hosted on Kaggle. Two ways to get it:

**Option A — KaggleHub (recommended, no manual download):**
```bash
pip install kagglehub
```

**Option B — Manual download:**
Download from Kaggle, unzip, and point `DATA_DIR` at the folder that contains
`seg_train/seg_train/<class>/...` and `seg_test/seg_test/<class>/...`.

The cell below tries KaggleHub first and falls back to a manual path.


In [ ]:
DATA_DIR = None

try:
    import kagglehub
    path = kagglehub.dataset_download("puneet6060/intel-image-classification")
    DATA_DIR = path
    print(f"Dataset downloaded to: {DATA_DIR}")
except Exception as e:
    print("KaggleHub download failed or is unavailable:", e)
    print("Set DATA_DIR manually to the folder containing seg_train/ and seg_test/.")
    DATA_DIR = "./intel-image-classification"  # <-- EDIT THIS if downloading manually

TRAIN_DIR = os.path.join(DATA_DIR, "seg_train", "seg_train")
TEST_DIR = os.path.join(DATA_DIR, "seg_test", "seg_test")

print("Train dir:", TRAIN_DIR)
print("Test dir:", TEST_DIR)


## 3. Data Preprocessing

- Resize every image to a uniform **150x150** pixels.
- Convert to tensor and **normalize** pixel values to `[-1, 1]` (mean/std = 0.5
  per channel is a common, dataset-agnostic normalization for RGB images).
- Carve a validation split out of the training set.


In [ ]:
IMG_SIZE = 150
BATCH_SIZE = 64
NUM_CLASSES = 6

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

full_train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=transform)
test_dataset = datasets.ImageFolder(root=TEST_DIR, transform=transform)

class_names = full_train_dataset.classes
print("Classes:", class_names)

val_fraction = 0.15
val_size = int(len(full_train_dataset) * val_fraction)
train_size = len(full_train_dataset) - val_size
train_dataset, val_dataset = random_split(
    full_train_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")


In [ ]:
# Quick visual sanity check of a batch
def show_batch(loader, n=6):
    images, labels = next(iter(loader))
    images = images[:n]
    labels = labels[:n]
    fig, axes = plt.subplots(1, n, figsize=(15, 3))
    for img, lbl, ax in zip(images, labels, axes):
        img = img.permute(1, 2, 0).numpy() * 0.5 + 0.5  # undo normalization
        ax.imshow(np.clip(img, 0, 1))
        ax.set_title(class_names[lbl])
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_batch(train_loader)


## 4. Model Architectures

### 4.1 Baseline ANN (Multi-Layer Perceptron)
Dense layers only — the image is flattened into a single vector, discarding
all 2D spatial structure.

### 4.2 CNN
Convolutional + max-pooling layers extract local, translation-invariant
spatial features before a small dense head does the classification.
Dropout is used for regularization.


In [ ]:
class ANN(nn.Module):
    def __init__(self, img_size=IMG_SIZE, num_classes=NUM_CLASSES):
        super().__init__()
        input_dim = img_size * img_size * 3
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.net(x)


class CNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 150 -> 75

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 75 -> 37

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 37 -> 18
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(128 * 18 * 18, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


ann_model = ANN().to(device)
cnn_model = CNN().to(device)

print(f"ANN trainable parameters: {count_parameters(ann_model):,}")
print(f"CNN trainable parameters: {count_parameters(cnn_model):,}")


## 5. Training Utilities

In [ ]:
def train_model(model, train_loader, val_loader, epochs=10, lr=1e-3):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    start_time = time.time()
    for epoch in range(epochs):
        # ---- Train ----
        model.train()
        running_loss, running_correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            running_correct += (outputs.argmax(1) == labels).sum().item()
            total += images.size(0)

        train_loss = running_loss / total
        train_acc = running_correct / total

        # ---- Validate ----
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)
                val_correct += (outputs.argmax(1) == labels).sum().item()
                val_total += images.size(0)

        val_loss /= val_total
        val_acc = val_correct / val_total

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"Epoch {epoch+1:02d}/{epochs} | "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    training_time = time.time() - start_time
    return history, training_time


def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += images.size(0)
    return correct / total


## 6. Train the ANN Baseline

In [ ]:
EPOCHS = 10

ann_model = ANN().to(device)
ann_history, ann_train_time = train_model(ann_model, train_loader, val_loader, epochs=EPOCHS)
ann_test_acc = evaluate(ann_model, test_loader)
ann_params = count_parameters(ann_model)

print(f"\nANN training time: {ann_train_time:.1f}s | Test accuracy: {ann_test_acc:.4f}")


## 7. Train the CNN

In [ ]:
cnn_model = CNN().to(device)
cnn_history, cnn_train_time = train_model(cnn_model, train_loader, val_loader, epochs=EPOCHS)
cnn_test_acc = evaluate(cnn_model, test_loader)
cnn_params = count_parameters(cnn_model)

print(f"\nCNN training time: {cnn_train_time:.1f}s | Test accuracy: {cnn_test_acc:.4f}")


## 8. Comparison Analysis

In [ ]:
comparison = pd.DataFrame({
    "Model": ["ANN (MLP)", "CNN"],
    "Trainable Parameters": [ann_params, cnn_params],
    "Training Time (s)": [round(ann_train_time, 1), round(cnn_train_time, 1)],
    "Validation Accuracy": [round(ann_history["val_acc"][-1], 4), round(cnn_history["val_acc"][-1], 4)],
    "Test Accuracy": [round(ann_test_acc, 4), round(cnn_test_acc, 4)],
})
comparison


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(ann_history["val_acc"], label="ANN")
axes[0].plot(cnn_history["val_acc"], label="CNN")
axes[0].set_title("Validation Accuracy per Epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].bar(comparison["Model"], comparison["Training Time (s)"], color=["#4C72B0", "#DD8452"])
axes[1].set_title("Total Training Time")
axes[1].set_ylabel("Seconds")

axes[2].bar(comparison["Model"], comparison["Trainable Parameters"], color=["#4C72B0", "#DD8452"])
axes[2].set_title("Trainable Parameters")
axes[2].set_ylabel("Count")

plt.tight_layout()
plt.show()


## 9. Discussion — Why CNNs Outperform ANNs on Spatial Image Data

- **Spatial structure is preserved.** The ANN immediately flattens the
  150x150x3 image into a single 67,500-element vector, discarding all
  information about which pixels are spatially adjacent. The CNN's
  convolutional filters slide over the 2D grid, so neighboring-pixel
  relationships (edges, textures, shapes) are captured directly.

- **Weight sharing → far fewer parameters for more capacity.** A dense
  layer connecting a flattened image to even a modest hidden layer
  needs *width × height × channels × hidden_units* parameters. A
  convolutional layer reuses the same small kernel (e.g. 3x3) across
  the entire image, so parameter count barely grows with image size,
  freeing up capacity for depth instead of brute-force width.

- **Translation invariance.** Because the same kernel is applied
  everywhere, a CNN recognizes a feature (e.g., a mountain ridge line)
  regardless of where it appears in the frame. An ANN must learn that
  pattern independently for every pixel position it might occur at.

- **Hierarchical feature learning.** Stacked conv + pooling layers build
  up from low-level features (edges, color gradients) in early layers
  to higher-level, more abstract concepts (textures, object parts) in
  later layers — a hierarchy that matches how visual scenes are
  actually composed.

- **Practical effect observed above.** The CNN typically reaches
  meaningfully higher validation/test accuracy than the ANN despite
  having comparable or fewer trainable parameters, and it does so more
  data-efficiently per epoch, because it isn't wasting capacity
  re-learning spatially redundant patterns.


## 10. Bonus: Inference Speed & CUDA / TensorRT Optimization

Two things affect deployed inference speed independently of model accuracy:
**how the model is executed** (framework/runtime) and **numerical precision**.

### 10.1 Quick inference benchmark (framework-level)
The cell below measures raw PyTorch inference latency/throughput for the CNN,
with and without `cudnn.benchmark` (which lets cuDNN auto-tune convolution
algorithms for fixed input shapes — a free win on GPU when input size is
constant, as it is here at 150x150).


In [ ]:
def benchmark_inference(model, loader, num_batches=20):
    model.eval()
    images, _ = next(iter(loader))
    images = images.to(device)

    # Warm-up
    with torch.no_grad():
        for _ in range(5):
            _ = model(images)
    if device.type == "cuda":
        torch.cuda.synchronize()

    start = time.time()
    with torch.no_grad():
        for _ in range(num_batches):
            _ = model(images)
    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.time() - start

    throughput = (num_batches * images.size(0)) / elapsed
    return elapsed / num_batches, throughput


if device.type == "cuda":
    torch.backends.cudnn.benchmark = False
    latency_off, tput_off = benchmark_inference(cnn_model, test_loader)

    torch.backends.cudnn.benchmark = True
    latency_on, tput_on = benchmark_inference(cnn_model, test_loader)

    print(f"cudnn.benchmark=False -> {latency_off*1000:.2f} ms/batch | {tput_off:.1f} img/s")
    print(f"cudnn.benchmark=True  -> {latency_on*1000:.2f} ms/batch  | {tput_on:.1f} img/s")
else:
    latency_cpu, tput_cpu = benchmark_inference(cnn_model, test_loader)
    print(f"Running on CPU (no CUDA available in this environment).")
    print(f"CPU inference -> {latency_cpu*1000:.2f} ms/batch | {tput_cpu:.1f} img/s")
    print("Re-run this cell on a CUDA-enabled machine to see the cudnn.benchmark comparison.")


### 10.2 Discussion: CUDA / TensorRT impact on inference speed

- **CUDA-level gains (free, no accuracy cost):**
  - `torch.backends.cudnn.benchmark = True` lets cuDNN pick the fastest
    convolution algorithm for a fixed input shape — usually a 10-30% latency
    reduction for CNNs like this one with a constant 150x150 input.
  - Batching inference requests (rather than one image at a time) amortizes
    kernel-launch overhead and keeps the GPU's compute units saturated.
  - Mixed precision (`torch.cuda.amp.autocast`, FP16) roughly halves memory
    bandwidth pressure and can nearly double throughput on GPUs with Tensor
    Cores, with negligible accuracy loss for a classification task like this.

- **TensorRT-level gains (compiled inference engine):**
  - Converting the trained PyTorch model to **TensorRT** (via `torch2trt`,
    `torch-tensorrt`, or the ONNX → TensorRT path) performs *layer fusion*
    (e.g., fusing Conv + BatchNorm + ReLU into a single kernel), kernel
    auto-tuning for the exact target GPU, and precision calibration
    (FP16 or INT8).
  - For a small CNN like this one, TensorRT conversion typically yields
    **2-5x lower latency** and correspondingly higher throughput compared
    to eager-mode PyTorch, with FP16/INT8 calibration giving the largest
    additional jump at a small, controllable accuracy cost.
  - The trade-off is engineering cost: TensorRT engines are tied to a
    specific GPU architecture and TensorRT version, and INT8 calibration
    requires a representative calibration dataset to preserve accuracy.
  - Net effect: none of this changes *what* the CNN learns — only how fast
    the already-trained weights can be executed at deployment time.

```python
# Illustrative only (requires torch-tensorrt / torch2trt + a CUDA GPU to run):
#
# import torch_tensorrt
# trt_model = torch_tensorrt.compile(
#     cnn_model,
#     inputs=[torch_tensorrt.Input((BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE))],
#     enabled_precisions={torch.float16},
# )
```


## 11. Conclusion

The CNN's convolutional and pooling layers exploit the 2D spatial structure
of images that the ANN discards by flattening, giving it higher accuracy,
better parameter efficiency, and translation-invariant feature extraction.
Framework-level (cuDNN autotuning, mixed precision) and compiler-level
(TensorRT) optimizations further improve deployed inference speed without
touching the learned weights, making the CNN both the more accurate and the
more deployment-friendly choice for this spatial image classification task.
